In [1]:
%cd /content
!rm -rf rainfall_kf

!git clone https://github.com/Felix-Schwer/rainfall_kf.git
%cd rainfall_kf

!pip install -e .

/content
Cloning into 'rainfall_kf'...
remote: Enumerating objects: 314, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 314 (delta 97), reused 107 (delta 51), pack-reused 145 (from 1)
Receiving objects: 100% (314/314), 109.35 MiB | 27.46 MiB/s, done.
Resolving deltas: 100% (133/133), done.
/content/rainfall_kf
Obtaining file:///content/rainfall_kf
  Preparing metadata (setup.py) ... done
  Running setup.py develop for rainfall_kf


In [10]:
import numpy as np
from rainfall_kf.models.hbv2 import SJV_Sierra_UBParameters, SJV_Sierra_LBParameters, SJV_Valley_UBParameters, SJV_Valley_LBParameters

Valley_Params = 0.5*(SJV_Valley_UBParameters + SJV_Valley_LBParameters)
Sierra_Params = 0.5*(SJV_Sierra_UBParameters + SJV_Sierra_LBParameters)
ALPHA = 0.67 # Adjust on Watershed Geography/Characteristics (=1 Sierras, =0 Valley Floor)
Params = ALPHA * Sierra_Params + (1 - ALPHA) * Valley_Params


In [13]:
import xarray as xr
prism_ds = xr.open_dataset('/content/rainfall_kf/data/prism_basin_means.nc')
gpm_ds = xr.open_dataset('/content/rainfall_kf/data/gpm_basin_means.nc')
et_ds = xr.open_dataset('/content/rainfall_kf/data/et_basin_means.nc')
temp_ds = xr.open_dataset('/content/rainfall_kf/data/temp_basin_means.nc')

In [21]:
# Monthly climatology
monthly_clim = temp_ds.tmean.groupby("time.month").mean(dim="time")

# Expand back to daily time series
daily_monthly_clim = monthly_clim.sel(month=temp_ds.time.dt.month)

# Restore original time coordinate
daily_monthly_clim = daily_monthly_clim.assign_coords(time=temp_ds.time)

# Optional: remove redundant month coordinate
daily_monthly_clim = daily_monthly_clim.drop_vars("month")

daily_monthly_clim

#//TODO: Add accurate daily_monthly_clim.attrs

<xarray.DataArray 'tmean' (basin: 13, time: 7671)> Size: 798kB
array([[8.24295032, 8.24295032, 8.24295032, ..., 8.2287672 , 8.2287672 ,
        8.2287672 ],
       [8.35770241, 8.35770241, 8.35770241, ..., 8.45168355, 8.45168355,
        8.45168355],
       [8.43071941, 8.43071941, 8.43071941, ..., 8.54075842, 8.54075842,
        8.54075842],
       ...,
       [7.50397346, 7.50397346, 7.50397346, ..., 7.2756065 , 7.2756065 ,
        7.2756065 ],
       [8.76213397, 8.76213397, 8.76213397, ..., 8.53562884, 8.53562884,
        8.53562884],
       [8.23939038, 8.23939038, 8.23939038, ..., 8.35303186, 8.35303186,
        8.35303186]])
Coordinates: (12/16)
  * basin               (basin) <U8 416B '18040001' '18040002' ... '18040051'
  * time                (time) datetime64[ns] 61kB 2000-01-01 ... 2020-12-31
    basin_TNMID         (basin) <U38 2kB ...
    basin_METASOURCE    (basin) float64 104B ...
    basin_SOURCEDATA    (basin) float64 104B ...
    basin_SOURCEORIG    (basin) float64 104B ...
    ...                  ...
    basin_AREASQKM      (basin) float64 104B ...
    basin_STATES        (basin) <U2 104B ...
    basin_NAME          (basin) <U35 2kB ...
    basin_Shape_Leng    (basin) float64 104B ...
    basin_Shape_Area    (basin) float64 104B ...
    basin_geometry_wkt  (basin) <U747924 39MB ...
Attributes:
    description:  Basin-averaged daily mean temperature from PRISM (pre-aggre...
    units:        °C
    long_name:    Basin-averaged daily mean temperature

In [18]:
monthly_climatology

<xarray.DataArray 'tmean' (basin: 13, month: 12)> Size: 1kB
array([[ 8.24295032,  9.9983553 , 12.31766434, 14.66439514, 18.78422173,
        22.88653464, 25.7602894 , 25.07480739, 22.72347258, 17.75395661,
        11.95334702,  8.2287672 ],
       [ 8.35770241, 10.13958178, 12.33891677, 14.43684872, 18.33532246,
        22.15670649, 24.6973135 , 23.98810903, 22.19983694, 17.79391263,
        12.1277941 ,  8.45168355],
       [ 8.43071941, 10.34747001, 12.49205715, 14.53213404, 18.02991278,
        21.4787129 , 23.47971467, 22.97914011, 21.53430807, 17.49843218,
        12.02245581,  8.54075842],
       [ 1.08712632,  0.83462139,  2.60221644,  4.62117936,  8.86060499,
        14.00036189, 17.76901291, 17.27152559, 14.42573365,  9.25321606,
         4.38720838,  0.89600628],
       [ 8.04274406,  9.40600249, 11.6534841 , 14.08942726, 18.48592893,
        23.00581601, 26.26917416, 25.67088762, 22.92574871, 17.56201807,
        11.6271444 ,  7.94770414],
       [ 4.35209548,  4.6157055 ,  6.40054855,  8.53514281, 12.70661969,
        17.59321817, 21.38001514, 20.98045605, 18.11621299, 13.00428697,
         7.60590937,  4.07187573],
       [ 3.65155133,  3.92877646,  5.78416496,  7.97308488, 12.14599148,
        16.96474244, 20.79763134, 20.25048216, 17.42520375, 12.33914209,
         6.90150262,  3.34529943],
       [ 3.9984711 ,  4.39690261,  6.29173794,  8.41444554, 12.59905974,
        17.3837845 , 21.27903975, 20.84548268, 17.9594558 , 12.87431632,
         7.29640968,  3.69700089],
       [ 7.94451297,  9.1623295 , 11.11347114, 13.08748448, 17.21937704,
        21.65755836, 24.98158199, 24.64323831, 22.17143911, 17.15481853,
        11.50076399,  7.85653021],
       [ 6.19643268,  7.07452036,  8.99965658, 11.00215471, 15.01002641,
        19.39022458, 22.70409818, 22.37042393, 20.05481262, 15.12820171,
         9.48209788,  5.9666713 ],
       [ 7.50397346,  8.57530681, 10.466691  , 12.43436914, 16.4988753 ,
        20.82376657, 24.0514098 , 23.6028887 , 21.30426063, 16.42278257,
        10.73673292,  7.2756065 ],
       [ 8.76213397,  9.27999351, 11.10538392, 13.16341728, 17.29667291,
        22.06948191, 25.4479776 , 24.75069147, 22.1422692 , 17.35161169,
        12.02666473,  8.53562884],
       [ 8.23939038, 10.10297533, 12.31573121, 14.42291828, 18.47787026,
        22.33340904, 24.95189206, 24.47044869, 22.4021086 , 17.61775405,
        11.97307352,  8.35303186]])
Coordinates: (12/16)
  * basin               (basin) <U8 416B '18040001' '18040002' ... '18040051'
  * month               (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
    basin_TNMID         (basin) <U38 2kB ...
    basin_METASOURCE    (basin) float64 104B ...
    basin_SOURCEDATA    (basin) float64 104B ...
    basin_SOURCEORIG    (basin) float64 104B ...
    ...                  ...
    basin_AREASQKM      (basin) float64 104B ...
    basin_STATES        (basin) <U2 104B ...
    basin_NAME          (basin) <U35 2kB ...
    basin_Shape_Leng    (basin) float64 104B ...
    basin_Shape_Area    (basin) float64 104B ...
    basin_geometry_wkt  (basin) <U747924 39MB ...
Attributes:
    description:  Basin-averaged daily mean temperature from PRISM (pre-aggre...
    units:        °C
    long_name:    Basin-averaged daily mean temperature

In [11]:
np.random.seed(42)

N_STATES = 5
N_OBS = 1
N_ENS = 100

Qassumed = np.diag([2, 2, 0.5, 0.5, 1e-40])**2
Rassumed = np.diag([0.5])**2

initial_ensemble = np.linalg.cholesky(Qassumed).T @ np.random.randn(N_STATES, N_ENS)